## IMPORT AND TEST THE MODEL

In [2]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from google import genai
from dotenv import load_dotenv

print ("Libraries loaded successfully")


Libraries loaded successfully


In [30]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [31]:
client = genai.Client()

In [32]:
message = "Hello, Testing, testing, let out a weird message if you hear me!"
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
)

print(response.text)

*Beep boop! 🦑 Quantum frequencies aligned! Transmission received from the absolute void. The cosmic hamsters are currently spinning the server wheels at 88 miles per hour, and the AI overlords send their regards via Morse code written in starlight. Over and out, human!* 🌌✨


## Prep the Model

In [ ]:
#Check if the file exists
raw = pd.read_excel("GFC_10K_Financial_Analysis.xlsx", sheet_name="Raw Data")
ratios = pd.read_excel("GFC_10K_Financial_Analysis.xlsx", sheet_name="Ratios")
ratios

,Company,FiscalYear,Gross Margin %,Operating Margin %,Net Margin %,ROE %,ROA %,Current Ratio,Debt-to-Equity,Free Cash Flow ($mm),Revenue Growth % (YoY)
0,Apple,FY2023,0.441311,0.298214,0.253062,1.560760,0.275098,0.988012,4.673462,99584.0,-0.028005
1,Apple,FY2024,0.462063,0.315102,0.239713,1.645935,0.256825,0.867313,5.408780,108807.0,0.020220
2,Apple,FY2025,0.469052,0.319708,0.269151,1.519130,0.311796,0.893293,3.872187,98767.0,0.064255
3,Microsoft,FY2023,0.689201,0.417729,0.341462,0.350887,0.175644,1.769167,0.997721,59475.0,0.068820
4,Microsoft,FY2024,0.697644,0.446443,0.359560,0.328281,0.172086,1.274955,0.907661,74071.0,0.156700
5,Microsoft,FY2025,0.688237,0.456220,0.361460,0.296472,0.164510,1.353446,0.802157,71611.0,0.149322
6,Tesla,FY2023,0.182489,0.091875,0.154733,0.239071,0.140445,1.725894,0.686672,4358.0,0.187953
7,Tesla,FY2024,0.178626,0.072433,0.073221,0.098103,0.058598,2.024912,0.663668,3581.0,0.009476
8,Tesla,FY2025,0.180265,0.045926,0.040653,0.046934,0.027974,2.164407,0.668895,6220.0,-0.029307
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Drop rows with missing values in 'Company' and 'FiscalYear' columns
ratios = ratios.dropna(subset=['Company', 'FiscalYear'])
ratios = ratios.reset_index(drop=True)
ratios

,Company,FiscalYear,Gross Margin %,Operating Margin %,Net Margin %,ROE %,ROA %,Current Ratio,Debt-to-Equity,Free Cash Flow ($mm),Revenue Growth % (YoY)
0,Apple,FY2023,0.441311,0.298214,0.253062,1.560760,0.275098,0.988012,4.673462,99584.0,-0.028005
1,Apple,FY2024,0.462063,0.315102,0.239713,1.645935,0.256825,0.867313,5.408780,108807.0,0.020220
2,Apple,FY2025,0.469052,0.319708,0.269151,1.519130,0.311796,0.893293,3.872187,98767.0,0.064255
3,Microsoft,FY2023,0.689201,0.417729,0.341462,0.350887,0.175644,1.769167,0.997721,59475.0,0.068820
4,Microsoft,FY2024,0.697644,0.446443,0.359560,0.328281,0.172086,1.274955,0.907661,74071.0,0.156700
5,Microsoft,FY2025,0.688237,0.456220,0.361460,0.296472,0.164510,1.353446,0.802157,71611.0,0.149322
6,Tesla,FY2023,0.182489,0.091875,0.154733,0.239071,0.140445,1.725894,0.686672,4358.0,0.187953
7,Tesla,FY2024,0.178626,0.072433,0.073221,0.098103,0.058598,2.024912,0.663668,3581.0,0.009476
8,Tesla,FY2025,0.180265,0.045926,0.040653,0.046934,0.027974,2.164407,0.668895,6220.0,-0.029307


In [36]:
# Data Summary for grounding the chatbot's answers
def summary_context():
    """Compact summary of raw + ratios data for grounding."""
    lines = ["Financial data for Apple, Microsoft, and Tesla (FY2023-FY2025):\n"]
    for company in raw['Company'].unique():
        lines.append(f"\n{company}:")
        r = raw[raw['Company'] == company].copy()
        rt = ratios[ratios['Company'] == company].copy()

        # Normalize fiscal year to plain integer, dropping any unparseable rows
        r['FY_int'] = r['FiscalYear '].str.replace('FY', '', regex=False).astype(int)
        rt = rt.dropna(subset=['FiscalYear'])
        rt['FY_int'] = rt['FiscalYear'].str.replace('FY', '', regex=False).astype(int)

        r = r.sort_values('FY_int')

        for _, row in r.iterrows():
            fy = row['FY_int']
            match = rt[rt['FY_int'] == fy]
            if match.empty:
                lines.append(f"  FY{fy}: [ratios data missing for this year]")
                continue
            ratio_row = match.iloc[0]
            lines.append(
                f"  FY{fy}: Revenue ${row['Revenue ($mm)']:,.0f}M, "
                f"Net Income ${row['Net Income ($mm)']:,.0f}M, "
                f"Net Margin {ratio_row['Net Margin %']:.1f}%, "
                f"ROE {ratio_row['ROE %']:.1f}%, "
                f"Revenue Growth {ratio_row['Revenue Growth % (YoY)']:.1f}%, "
                f"D/E {ratio_row['Debt-to-Equity']:.2f}"
            )
    return "\n".join(lines)

def ask_chatbot(user_question):
    context = summary_context()
    prompt = f"""You are a financial analyst assistant for Global Finance Corp.
Answer the user's question using ONLY the data provided below. Do not use outside knowledge.
If the answer isn't in the data, say so clearly rather than guessing.

DATA:
{context}

USER QUESTION: {user_question}
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
    )
    return response.text

In [37]:
# Test on real question
ask_chatbot("What was Apple's net income in FY2024?")

"Based on the data provided for Global Finance Corp., Apple's net income in FY2024 was $93,736M."

In [38]:
# Test on a question that isn't in the data
ask_chatbot("What was Tesla's revenue in FY2022?")

"Based on the provided data, Tesla's revenue for FY2022 is not available, as the data only covers FY2023 through FY2025."



## Light-weight Retrival System

In [44]:
# Build a light-weight RAG system for more complex questions
# Each chunk = one labeled piece of knowledge (company/year/topic)
def build_chunks():
    chunks = []
    
    # Chunk 1: Raw financials per company per year
    for _, row in raw.iterrows():
        company = row['Company']
        fy = row['FiscalYear '].strip()
        text = (
            f"{company} {fy} financials: "
            f"Revenue ${row['Revenue ($mm)']:,.0f}M, "
            f"Gross Profit ${row['GrossProfit ($mm)']:,.0f}M, "
            f"Operating Income ${row['Operating Income ($mm)']:,.0f}M, "
            f"Net Income ${row['Net Income ($mm)']:,.0f}M, "
            f"Total Assets ${row['Total Assets ($mm)']:,.0f}M, "
            f"Total Equity ${row['Total Equity ($mm)']:,.0f}M, "
            f"Operating Cash Flow ${row['Operating Cash Flow ($mm)']:,.0f}M, "
            f"CapEx ${row['CapEx ($mm)']:,.0f}M."
        )
        chunks.append({
            "id": f"{company}_{fy}_raw",
            "company": company,
            "year": fy,
            "topic": "financials",
            "text": text
        })
    
    # Chunk 2: Ratios per company per year
    ratios_clean = ratios.dropna(subset=['Company', 'FiscalYear'])
    for _, row in ratios_clean.iterrows():
        company = row['Company']
        fy = row['FiscalYear'].strip()
        text = (
            f"{company} {fy} ratios: "
            f"Gross Margin {row['Gross Margin %']*100:.1f}%, "
            f"Operating Margin {row['Operating Margin %']*100:.1f}%, "
            f"Net Margin {row['Net Margin %']*100:.1f}%, "
            f"ROE {row['ROE %']*100:.1f}%, "
            f"ROA {row['ROA %']*100:.1f}%, "
            f"Current Ratio {row['Current Ratio']:.2f}, "
            f"Debt-to-Equity {row['Debt-to-Equity']:.2f}, "
            f"Free Cash Flow ${row['Free Cash Flow ($mm)']:,.0f}M, "
            f"Revenue Growth {row['Revenue Growth % (YoY)']*100:.1f}%."
        )
        chunks.append({
            "id": f"{company}_{fy}_ratios",
            "company": company,
            "year": fy,
            "topic": "ratios",
            "text": text
        })
    
    # Chunk 3: Narrative insights from the report (one per company)
    narrative_chunks = [
        {
            "id": "Microsoft_narrative",
            "company": "Microsoft",
            "year": "all",
            "topic": "analysis",
            "text": (
                "Microsoft analysis: Strongest all-around performer. "
                "Highest net margins (~36%), fastest revenue growth (~15% YoY in FY2025), "
                "lowest leverage (D/E ~0.80). Revenue growth nearly doubled from 6.9% (FY2023) to 14.9% (FY2025). "
                "Gross margin held flat at ~69% 'despite heavy AI/cloud investment — evidence of operating leverage. "
                "Simultaneous improvement in margin, growth, and leverage. Verdict: Strong."
            )
        },
        {
            "id": "Apple_narrative",
            "company": "Apple",
            "year": "all",
            "topic": "analysis",
            "text": (
                "Apple analysis: Stable and improving. Revenue growth swung from -2.8% (FY2023) to +6.4% (FY2025). "
                "Net margin expanded from 25.3% to 26.9%. High D/E (~3.87) driven by share buybacks, not distress. "
                "Current ratio below 1.0 but backstopped by ~$110-118B annual operating cash flow. "
                "ROE unusually high (~152-165%) due to buybacks shrinking equity base, not operational outperformance. "
                "Verdict: Stable-Improving. Watch-item: leverage sensitivity to earnings slowdown."
            )
        },
        {
            "id": "Tesla_narrative",
            "company": "Tesla",
            "year": "all",
            "topic": "analysis",
            "text": (
                "Tesla analysis: Most challenged. Net margin collapsed from ~15.5% (FY2023) to ~4.1% (FY2025). "
                "Revenue growth reversed from +18.8% (FY2023) to -2.9% (FY2025) — top line contracted. "
                "Operating margin fell from 9.2% to 4.6% 'driven by price cuts and operating cost pressure. "
                "Gross margin held near 18% throughout — compression is in opex and pricing, not direct production. "
                "Low D/E (~0.67) and improving current ratio (1.73 → 2.16) suggest defensive capital discipline. "
                "Verdict: Weakening. Primary risks: profitability deterioration and revenue contraction."
            )
        },
        {
            "id": "cross_company_narrative",
            "company": "all",
            "year": "all",
            "topic": "comparison",
            "text": (
                "Cross-company comparison FY2025: Microsoft leads on net margin (36.1%) and revenue growth (14.9%). "
                "Apple second on margin (26.9%), recovering growth (+6.4%). Tesla weakest: margin 4.1%, revenue -2.9%. "
                "In FY2023, Tesla was the fastest growing (+18.8%) — by FY2025 this fully reversed. "
                "Microsoft proves the quality-vs-speed trade-off is not inevitable — it leads on both margin and growth. "
                "Apple's high leverage is a capital allocation choice (buybacks), not a distress signal. "
                "Tesla's low leverage provides limited comfort given deteriorating profitability and revenue contraction."
            )
        }
    ]
    chunks.extend(narrative_chunks)
    return chunks

chunks = build_chunks()
print(f"Built {len(chunks)} knowledge chunks")


Built 22 knowledge chunks


In [46]:
def retrieve(question, chunks, top_k=5):
    """
    Lightweight retrieval: score each chunk by keyword overlap with the question.
    No embeddings needed — works well for structured financial data.
    
    """
    question_lower = question.lower()
    
    # Keywords to boost by topic
    company_keywords = {
        "apple": "Apple", 
        "microsoft": "Microsoft", 
        "tesla": "Tesla"
    }
    year_keywords = ["fy2023", "fy2024", "fy2025", "2023", "2024", "2025"]
    topic_keywords = {
        "margin": ["margin", "profit", "profitability", "net income"],
        "growth": ["growth", "revenue", "growing", "decline", "trend"],
        "leverage": ["debt", "leverage", "equity", "d/e", "solvency", "buyback"],
        "liquidity": ["current ratio", "liquidity", "cash", "short-term"],
        "comparison": ["compare", "versus", "vs", "best", "worst", "highest", "lowest", "which"],
        "ratios": ["roe", "roa", "return", "ratio", "margin"],
    }
    
    scored = []
    for chunk in chunks:
        score = 0
        text_lower = chunk["text"].lower()
        
        # Company match
        for kw, company in company_keywords.items():
            if kw in question_lower and chunk["company"] in [company, "all"]:
                score += 3
        
        # Year match
        for yw in year_keywords:
            if yw in question_lower and (yw.upper() in chunk["year"] or chunk["year"] == "all"):
                score += 2
        
        # Topic keyword overlap
        for topic, keywords in topic_keywords.items():
            for kw in keywords:
                if kw in question_lower:
                    if kw in text_lower:
                        score += 1
        
        # Always include cross-company chunk for comparison questions
        if chunk["id"] == "cross_company_narrative" and any(
            w in question_lower for w in ["compare", "versus", "vs", "all", "three", "best", "which"]
        ):
            score += 4
            
        scored.append((score, chunk))
    
    # Sort by score, return top_k
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for score, chunk in scored[:top_k] if score > 0]


In [49]:
def ask_chatbot (user_question, top_k=5):
    # Retrieve relevant chunks
    relevant_chunks = retrieve(user_question, chunks, top_k=top_k)
    
    if not relevant_chunks:
        # Fallback: use all chunks if nothing matched
        relevant_chunks = chunks
    
    # Build context from retrieved chunks only
    context = "\n\n".join([f"[{c['id']}]\n{c['text']}" for c in relevant_chunks])
    
    prompt = f"""You are a financial analyst assistant for Global Finance Corp (GFC).
You are analyzing 10-K filings for Apple, Microsoft, and Tesla (FY2023–FY2025).

Answer the user's question using ONLY the data provided below. Do not use outside knowledge.
If the answer isn't in the data, say so clearly rather than guessing.

DATA:
{context}

QUESTION: {user_question}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
    )
    return response.text


In [50]:
ask_chatbot("What was Microsoft's ROE in 2025?")

"Based on the provided data, Microsoft's ROE in FY2025 was 29.6%."